# Subconsultas y CTEs

## Subconsultas en WHERE

In [10]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute(
    '''CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, department TEXT, salary REAL)''')
cursor.executemany('INSERT INTO employees VALUES (?,?,?,?)', [
    (1, 'Alice', 'Engineering', 75000), (2, 'Bob', 'Marketing', 55000),
    (3, 'Carol', 'Engineering', 82000), (4, 'David', 'HR', 48000),
    (5, 'Eva', 'Marketing', 61000), (6, 'Frank', 'Engineering', 79000),
    (7, 'Grace', 'HR', 52000),
])
conn.commit()

# print("Empleados con salario > promedio:")
# cursor.execute("")
# avg = cursor.fetchone()[0]
# print(f"  (Promedio: ${avg:,.0f})")

# Subconsulta escalar: empleados con salario > promedio
cursor.execute("""
    Select 
        name, salary, department
    from 
        employees
    where 
        salary > (select AVG(salary) from employees)
    order by 
        salary DESC
""")
for row in cursor.fetchall():
    print(f" | {row[0]:>8} | ${row[1]:<10} | {row[2]:<15} |")

# Subconsulta con IN: depts con salario promedio > 60k
cursor.execute("""
               Select name, department, salary 
               FROM employees
               WHERE department IN (
                   select department
                   from employees
                   group by department
                   having AVG(salary) > 60000
               )
               order by salary desc
""")
print("\nEmpleados en depts con AVG > 60k:")
for row in cursor.fetchall():
    print(row)
    # print(f"  ${row[0]:<8} | ${row[1]:<12} | ${row[2]:<15}")

 |    Carol | $82000.0    | Engineering     |
 |    Frank | $79000.0    | Engineering     |
 |    Alice | $75000.0    | Engineering     |

Empleados en depts con AVG > 60k:
('Carol', 'Engineering', 82000.0)
('Frank', 'Engineering', 79000.0)
('Alice', 'Engineering', 75000.0)
